## Setup

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 72
from IPython.display import display
import numpy as np
from scipy.sparse import lil_matrix
import os
import pandas as pd

MOVIE_DATA_CSV= 'movie_data.csv'

In [2]:
df = pd.DataFrame()
df = pd.read_csv(MOVIE_DATA_CSV)    

# Sanity
print(f'#rows={len(df)} #columns={len(df.columns)}')
df.head()

#rows=50000 #columns=2


,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0
3,hi for all the people who have seen this wonde...,1
4,"I recently bought the DVD, forgetting just how...",0


In [3]:
Reviews = [_ for _ in df.review if _ != 'Unknown']

# sanity
print(len(Reviews))
print(Reviews[0:3])

50000
['In 1974, the teenager Martha Moxley (Maggie Grace) moves to the high-class area of Belle Haven, Greenwich, Connecticut. On the Mischief Night, eve of Halloween, she was murdered in the backyard of her house and her murder remained unsolved. Twenty-two years later, the writer Mark Fuhrman (Christopher Meloni), who is a former LA detective that has fallen in disgrace for perjury in O.J. Simpson trial and moved to Idaho, decides to investigate the case with his partner Stephen Weeks (Andrew Mitchell) with the purpose of writing a book. The locals squirm and do not welcome them, but with the support of the retired detective Steve Carroll (Robert Forster) that was in charge of the investigation in the 70\'s, they discover the criminal and a net of power and money to cover the murder.<br /><br />"Murder in Greenwich" is a good TV movie, with the true story of a murder of a fifteen years old girl that was committed by a wealthy teenager whose mother was a Kennedy. The powerful and ric

## Problem 1 

The following code tokenizes the reviews in the movies data set into a list-of-list of tokens. Prior to tokenization, it removes the three HTML tags I saw in the data ```<br />, <i />, and <em />```.

In [4]:
from nltk import regexp_tokenize
from nltk.corpus import stopwords
import nltk

import re

HTML_TAGS = [
    r'<\s*br\s*/?\s*>',
    r'<\s*/?\s*i\s*>',
    r'<\s*/?\s*em\s*>',
]

def strip_known_html(text):
    for pat in HTML_TAGS:
        text = re.sub(pat, ' ', text, flags=re.IGNORECASE)
    return text

# from the sklearn API the default token_pattern='(?u)\\b\\w\\w+\\b'
TOKEN_PATTERN= r'(?u)\b[a-zA-Z]\w+\b'

Reviews_tok = [
    regexp_tokenize(strip_known_html(r).lower(), TOKEN_PATTERN)
    for r in Reviews
]

print(Reviews_tok[0:3])

[['in', 'the', 'teenager', 'martha', 'moxley', 'maggie', 'grace', 'moves', 'to', 'the', 'high', 'class', 'area', 'of', 'belle', 'haven', 'greenwich', 'connecticut', 'on', 'the', 'mischief', 'night', 'eve', 'of', 'halloween', 'she', 'was', 'murdered', 'in', 'the', 'backyard', 'of', 'her', 'house', 'and', 'her', 'murder', 'remained', 'unsolved', 'twenty', 'two', 'years', 'later', 'the', 'writer', 'mark', 'fuhrman', 'christopher', 'meloni', 'who', 'is', 'former', 'la', 'detective', 'that', 'has', 'fallen', 'in', 'disgrace', 'for', 'perjury', 'in', 'simpson', 'trial', 'and', 'moved', 'to', 'idaho', 'decides', 'to', 'investigate', 'the', 'case', 'with', 'his', 'partner', 'stephen', 'weeks', 'andrew', 'mitchell', 'with', 'the', 'purpose', 'of', 'writing', 'book', 'the', 'locals', 'squirm', 'and', 'do', 'not', 'welcome', 'them', 'but', 'with', 'the', 'support', 'of', 'the', 'retired', 'detective', 'steve', 'carroll', 'robert', 'forster', 'that', 'was', 'in', 'charge', 'of', 'the', 'investigat

## Problem 2

The following code pulls in the sentiments and counts the number of sentences for each review then adds that number to the total sentence count corresponding to the review's sentiment.

In [5]:
from nltk.tokenize import sent_tokenize

Sentiments = [_ for _ in df.sentiment if _ != 'Unknown']

HTML_TAGS = [
    r'<\s*br\s*/?\s*>',
    r'<\s*/?\s*i\s*>',
    r'<\s*/?\s*em\s*>',
]

def strip_known_html(text):
    for pat in HTML_TAGS:
        text = re.sub(pat, ' ', text, flags=re.IGNORECASE)
    return text

# from the sklearn API the default token_pattern='(?u)\\b\\w\\w+\\b'
TOKEN_PATTERN= r'(?u)\b[a-zA-Z]\w+\b'

# sanity
print(len(Sentiments))
print(Sentiments[0:3])

sent_counts = {0: 0, 1: 0}

for review, sentiment in zip(Reviews, Sentiments):
    sents = sent_tokenize(strip_known_html(review))
    sent_counts[sentiment] += len(sents)

print(sent_counts)


50000
[1, 0, 0]
{0: 316400, 1: 300355}


There are:
- <b>316,400</b> sentences in reviews with <b>sentiment 0</b>
- <b>300,355</b> sentences in reviews with <b>sentiment 1</b>

For this analysis, it <b>does not matter</b> if the sentences are coming from different users because we're just trying to sort out how many sentences per sentiment without regard to users. 

If we were doing an analysis based on some user demographic, or if we wanted to take into account the impact of, say, highly prolific effusive users on the data, it would matter which users the sentiments were coming from.

## Problem 3



Iterate through the reviews and sentiments, stripping html, but leaving filler words for now. 

In [6]:
import re

stats = {
    0: {
    'sentence_count': 0,
    'total_words': 0,
    'min_words': 99999999,
    'max_words': 0
    }, 
    1: {
    'sentence_count': 0,
    'total_words': 0,
    'min_words': 99999999,
    'max_words': 0
    }
}

reviews_sentences_tokens = {0:[],1:[]}

for review, sentiment in zip(Reviews, Sentiments):
    # Strip HTML from the review and tokenize it into sentences
    sentences = sent_tokenize(strip_known_html(review))     
    
    # Increment the number of sentences for this review
    stats[sentiment]['sentence_count'] += len(sentences)     
    
    # Break each sentence into words
    sentence_tokens = []
    review_word_count = 0
    for sentence in sentences:
        words = [
            w
            for w in regexp_tokenize(sentence.lower(), TOKEN_PATTERN)
        ]
        
        if len(words)==0: 
            continue

        sentence_tokens.append(words)

        # Keep track of the total words
        stats[sentiment]['total_words'] += len(words)
        review_word_count += len(words)        
    
    reviews_sentences_tokens[sentiment].append(sentence_tokens)

    # Track min and max words per review
    stats[sentiment]['min_words'] = min(review_word_count, stats[sentiment]['min_words'])
    stats[sentiment]['max_words'] = max(review_word_count, stats[sentiment]['max_words'])
    

# Sanity check
print(reviews_sentences_tokens[0][0:3])
print(reviews_sentences_tokens[1][0:3])
print(reviews_sentences_tokens[0][0])
print(reviews_sentences_tokens[0][0][0])
print(reviews_sentences_tokens[0][0][0][0])

print(stats)

[[['ok', 'so'], ['really', 'like', 'kris', 'kristofferson', 'and', 'his', 'usual', 'easy', 'going', 'delivery', 'of', 'lines', 'in', 'his', 'movies'], ['age', 'has', 'helped', 'him', 'with', 'his', 'soft', 'spoken', 'low', 'energy', 'style', 'and', 'he', 'will', 'steal', 'scene', 'effortlessly'], ['but', 'disappearance', 'is', 'his', 'misstep'], ['holy', 'moly', 'this', 'was', 'bad', 'movie'], ['must', 'give', 'kudos', 'to', 'the', 'cinematography', 'and', 'and', 'the', 'actors', 'including', 'kris', 'for', 'trying', 'their', 'darndest', 'to', 'make', 'sense', 'from', 'this', 'goofy', 'confusing', 'story'], ['none', 'of', 'it', 'made', 'sense', 'and', 'kris', 'probably', 'didn', 'understand', 'it', 'either', 'and', 'he', 'was', 'just', 'going', 'through', 'the', 'motions', 'hoping', 'someone', 'would', 'come', 'up', 'to', 'him', 'and', 'tell', 'him', 'what', 'it', 'was', 'all', 'about'], ['don', 'care', 'that', 'everyone', 'on', 'this', 'movie', 'was', 'doing', 'out', 'of', 'love', 'fo

Sentiment 0
Total words: 5,439,393
Min words in a review: 6
Max words in a review: 1455

Sentiment 1
Total words: 5,534,065
Min words in a review: 10
Max words in a review: 2366

*Note: These results are without stripping out stop words which we do later for problem 4.*


## Problem 4

### Setup
We use the Markov class from the Markov Models notebook given in Canva

In [7]:
class Markov:  # Uses a sparse matrix to handle probabilities
    def __init__(self, order=1):
        self.voc, self.vocindex, self.vocindexrev = None, None, None
        self.conds, self.condsindex, self.condsindexrev = None, None, None
        self.Pjoint = None  # Joint probabilities
        self.Pvoc = None  # marginal P for vocabulary
        self.Pconds = None  # marginal P for conditionings
        self.Pconditional = None  # Conditional probabilities, Markov model
        self.order = order

    def get_vocabulary(self, _toks):  # input as list of tokens, 1-gram
        voc = set()
        for _ in _toks:
            voc.update(_)
        self.voc = sorted(voc)
        self.vocindex = {v:i for i, v in enumerate(self.voc)}
        self.vocindexrev = {i:v for i, v in enumerate(self.voc)}
    
    def get_conds(self, _toks):  # conditioning events, n-1-gram
        conds = set()
        for tok in _toks:
            for i in range(len(tok)-self.order):  # boundary condition
                cond = " ".join(tok[i:i+self.order])
                conds.update([cond])
        self.conds = sorted(conds)
        self.condsindex = {cond:i for i, cond in enumerate(self.conds)}
        self.condsindexrev = {i:cond for i, cond in enumerate(self.conds)}
    
    def get_counts(self, _toks):  # build the P
        M, N = len(self.vocindex), len(self.condsindex)
        pc = lil_matrix((N,M), dtype=np.float32)
        # pc = np.zeros((N,M), dtype=np.float32)
        for tok in _toks:
            for i in range(len(tok)-self.order-1):  # boundary condition
                cond = " ".join(tok[i:i+self.order])
                voc = tok[i+self.order]
                i, j = self.condsindex[cond], self.vocindex[voc]
                pc[i,j] += 1
        return pc

    def fit(self, _toks):
        self.get_vocabulary(_toks)
        self.get_conds(_toks)
        pc = self.get_counts(_toks)
        # joint P, make it into probabilities
        self.Pjoint = pc / np.sum(pc)
        # marginal P for vocabulary and conds
        self.Pvoc = np.array(np.sum(self.Pjoint,axis=0)).squeeze()
        self.Pconds = np.array(np.sum(self.Pjoint,axis=1)).squeeze()
        # conditional P
        sm = np.array(pc.sum(axis=1)).squeeze()
        sm[sm==0] = 1  # handle divide by zero
        self.Pconditional = lil_matrix(pc / sm[:,None])
        # sanity
        print(f'Size of vocabulary={len(self.voc)}, N={len(self.conds)}, Pjoint.shape={self.Pjoint.shape}')
        return self

Check the P matrix size for the entire data set which includes stop words

In [8]:
markov = Markov(order=1).fit(Reviews_tok)

Size of vocabulary=100243, N=99658, Pjoint.shape=(99658, 100243)


For the entire set, removing html tags and numbers, keeping stop words and both sentiments together, the P matrix is 

$\displaystyle 99658 \times 100243 \approx 10^{10}$

Now we group by sentiment and remove stop words

In [9]:
# These are needed for stopwords
nltk.download("punkt")
nltk.download("punkt_tab")

nltk.download('stopwords')

STOP_WORDS = set(stopwords.words('english'))

tokens_by_sentiment = {0: [], 1: []}

def clean_and_tokenize_review(review: str) -> list[str]:
    text = strip_known_html(review)
    tokens = regexp_tokenize(text.lower(), TOKEN_PATTERN)  
    tokens = [t for t in tokens if t not in STOP_WORDS]
    return tokens

for review, sentiment in zip(Reviews, Sentiments):
    toks = clean_and_tokenize_review(review)
    if len(toks) == 0:
        continue  # skip empty reviews after filtering
    tokens_by_sentiment[int(sentiment)].append(toks)
    
markov0_1 = Markov(order=1).fit(tokens_by_sentiment[0])
markov1_1 = Markov(order=1).fit(tokens_by_sentiment[1])

[nltk_data] Downloading package punkt to /Users/jeff/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/jeff/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/jeff/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Size of vocabulary=71539, N=71168, Pjoint.shape=(71168, 71539)
Size of vocabulary=74531, N=74136, Pjoint.shape=(74136, 74531)


With stop words removed and grouping down by sentiment the size of the P matrix is

Sentiment 0: $71168 \times 71539 \approx 5.1 \times 10^9$

Sentiment 1: $74136 \times 74531 \approx 5.5 \times 10^9$

## Problem 5

For each sentiment group, we find the index of the word "good" then look up the probability in Pvoc

In [10]:
def word_probability(markov, word: str) -> float:
    word = word.lower()
    idx = markov.vocindex.get(word)
    return float(markov.Pvoc[idx])

p_good_0 = word_probability(markov0_1, "good")
print(f"{p_good_0:.6f}")

p_good_1 = word_probability(markov1_1, "good")
print(f"{p_good_1:.6f}")

0.005087
0.005010


The probability of the word "good" is

Sentiment 0: 0.5087%

Sentiment 1: 0.5010%

## Problem 6

We switch to an order 3 model because the context is 3 tokens, then we look up the probability in Pconditional

In [11]:
markov0_3 = Markov(order=3).fit(tokens_by_sentiment[0])
markov1_3 = Markov(order=3).fit(tokens_by_sentiment[1])

Size of vocabulary=71539, N=2645267, Pjoint.shape=(2645267, 71539)
Size of vocabulary=74531, N=2763821, Pjoint.shape=(2763821, 74531)


In [12]:
cond = 'one best movies'
word = 'ever'

def getprobability(_model,_cond,_w):
    i = _model.condsindex[_cond]
    j = _model.vocindex[_w]
    return _model.Pconditional[i, j]

print(f"Sentiment 0")
print(f"{cond} {word}  (P={getprobability(markov0_3, cond, word):.5f})")

print(f"Sentiment 1")
print(f"{cond} {word}  (P={getprobability(markov1_3, cond, word):.5f})")

Sentiment 0
one best movies ever  (P=0.66667)
Sentiment 1
one best movies ever  (P=0.37226)


For sentiment 0, the probability is <b>66.67%</b>

For sentiment 1, the probability is <b>37.22%</b>

## Problem 7

We go back to using the order 1 model since the context is 1 token

In [13]:
SEQ = 'worst'

def getwords(_model, _w):
    i1 = _model.condsindex[_w]
    arr = _model.Pconditional[i1,:].toarray().squeeze()  # argsort will not work on lil_matrix
    ix = np.argsort(arr)[::-1]
    return [(_model.vocindexrev[ix[_]], _model.Pconditional[_model.condsindex[_w],ix[_]]) for _ in range(10)]

result = getwords(markov0_1, SEQ)

print(f"Sentiment 0")
print(f"{SEQ} {result[0][0]}  (P={result[0][1]:.5f})")
print(f"{SEQ} {result[1][0]}  (P={result[1][1]:.5f})")
print(f"{SEQ} {result[2][0]}  (P={result[2][1]:.5f})")

result = getwords(markov1_1, SEQ)

print(f"Sentiment 1")
print(f"{SEQ} {result[0][0]}  (P={result[0][1]:.5f})")
print(f"{SEQ} {result[1][0]}  (P={result[1][1]:.5f})")
print(f"{SEQ} {result[2][0]}  (P={result[2][1]:.5f})")


Sentiment 0
worst movie  (P=0.14764)
worst movies  (P=0.08437)
worst film  (P=0.07920)
Sentiment 1
worst part  (P=0.04318)
worst movie  (P=0.04091)
worst thing  (P=0.03864)


## Problem 8

We use the order 3 model since the context is 3 tokens

In [ ]:

SEQ = 'worst movie ever'

result = getwords(markov0_3, SEQ)

print(f"Sentiment 0")
print(f"{SEQ} {result[0][0]}  (P={result[0][1]:.5f})")
print(f"{SEQ} {result[1][0]}  (P={result[1][1]:.5f})")
print(f"{SEQ} {result[2][0]}  (P={result[2][1]:.5f})")

result = getwords(markov1_3, SEQ)

print(f"Sentiment 1")
print(f"{SEQ} {result[0][0]}  (P={result[0][1]:.5f})")
print(f"{SEQ} {result[1][0]}  (P={result[1][1]:.5f})")
print(f"{SEQ} {result[2][0]}  (P={result[2][1]:.5f})")


Sentiment 0
worst movie ever seen  (P=0.57710)
worst movie ever made  (P=0.15654)
worst movie ever saw  (P=0.02336)
Sentiment 1
worst movie ever seen  (P=0.40000)
worst movie ever since  (P=0.10000)
worst movie ever idiots  (P=0.10000)


: 

## Problem 9 

My company specializes in optimization, particularly **Mixed Integer Linear Programming (MILP)**.  

Below are three generative-AI project ideas related to optimization:

### 1. Solver Log Copilot
MIP, LP, and heuristic solver logs are often long and noisy, making it difficult to identify the root causes of poor performance (e.g., presolve issues, degeneracy, cuts, branching behavior, numerical instability, memory constraints).

The goal of this project is to train and evaluate a system that transforms raw solver logs into:
- Structured summaries (solver phases, key events, and bottlenecks)
- Actionable diagnostics (e.g., “the solver likely stalled due to degeneracy”)
- Recommended parameter or model adjustments

---

### 2. Cut and Heuristic Generator for MILP
Well-chosen cutting planes, primal heuristics, and warm starts can significantly reduce solve times, but designing them manually is challenging and problem-dependent.

This project explores using generative models to propose:
- Candidate cutting planes (e.g., cut families and variable subsets)
- Primal heuristics for solution construction
- Warm starts based on patterns learned from similar problem instances

---

### 3. Natural Language to MIP Generator
Translating real-world problem descriptions into correct and efficient MIP formulations is error-prone and typically requires expert knowledge.

This project aims to build a generative system that:
- Takes a natural-language optimization problem description as input
- Produces a structured MIP formulation (decision variables, objective function, and constraints)
- Outputs both human-readable LaTeX and solver-ready representations (e.g., LP or MPS files)
